# General

For more informations, see the documentation *LLM and GenAI*.

# Import & Configs

In [1]:
import json
import pandas as pd

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [4]:
%load_ext autoreload
%autoreload 2

from src.retrieval.retriever import MedicalRetriever
from src.pipeline.medical_assistant import ask_medical_assistant

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Questions Set

In [5]:
with open(
    "../data/evaluations/gold_questions.json"
) as f:

    questions = json.load(f)

questions

[{'question': 'What is glioblastoma?'},
 {'question': 'How is glioblastoma prognosis evaluated?'},
 {'question': 'What MRI techniques are used for glioblastoma?'},
 {'question': 'What is peritumoral edema?'},
 {'question': 'What is pseudoprogression?'},
 {'question': 'How is tumor progression detected?'},
 {'question': 'What biomarkers are associated with glioblastoma?'},
 {'question': 'What are the limitations of MRI in glioma diagnosis?'},
 {'question': 'How is overall survival measured?'},
 {'question': 'What treatments are commonly used for glioblastoma?'}]

# Retrieval Evaluation

In [6]:
retriever = MedicalRetriever()

/home/jeremy/Documents/dev/LLM_RAG/Medical_assistant/.ma_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
results_retriever = []

for item in questions:
    
    question = item["question"]
    print(f'{question:-^80}')
    #output = retriever.retrieve(question)
    output = retriever.retrieve_diverse_articles(question)
    results_retriever.append(output)
    print('\n')
    for key, val in output.items():
        #print(f'--- {key}:')
        if key == 'documents':
            for i, doc in enumerate(val[0]):
                print(f'[{output['ids'][0][i]}]\n{doc}\n')
        else:
            #print(f'{val}')
            pass
        #print('\n')
    print("\n\n")

-----------------------------What is glioblastoma?------------------------------
Original query: What is glioblastoma?
Processed query: glioblastoma


[42135047_3]
Brain gliomas are among the most common primary brain tumors of the central nervous system, encompassing a wide spectrum of tumor grades and diverse biological behaviors. Key aspects of their clinical management include accurate diagnosis and grading, tumor extent delineation, preoperative evaluation and radiation therapy target planning, as well as differentiation between post-treatment recurrence and treatment-related changes.

[42000417_0]
Glioblastoma IDH-wildtype: An integrative review of pathophysiological mechanisms, diagnostic innovations, and emerging therapeutic modalities. The most dangerous and fatal primary brain tumor in adults is glioblastoma multiforme/IDH wild-type glioblastoma (GBM), which is progressive, diffuse, and unresponsive to conventional treatment.

[41945366_1]
Stereotactic biopsy performed at the

### V1

| Question  | Retrieval | Comments |
|-----|:-----:|-----|
| Q1 | 4 |  |
| Q2 | 3 |  |
| Q3 | 2 |  |
| Q4 | 3 | Confusion btw glioma & glioblastoma |
| Q5 | 1 | Not Relevant | 
| Q6 | 2 | Relevance ? | 
| Q7 | 3 |  |
| Q8 | 2 | Only doc 1 is relevant |
| Q9 | 2 | Doc 1 : not relevant |
| Q10 | 1 | Confusion btw glioma & glioblastoma |

With:
- 1 = bad
- 5 = excellent

**Definitions:**
- **Retrieval:** Have the correct documents been found?

Identified Limitations:
- Chunks that are too specialized.
- Chunks at the wrong level
- Diversification issues

### V2

| Question  | Retrieval | Comments |
|-----|:-----:|-----|
| Q1 | 4 | 1 doc too large, others good |
| Q2 | 3 | Correct, but not ideal |
| Q3 | 4 |  |
| Q4 | 4 |  |
| Q5 | 3 | Doc 2 : weak | 
| Q6 | 2 | progression more than detection | 
| Q7 | 4 |  |
| Q8 | 2 | Only doc 1 is relevant |
| Q9 | 1 | Not relevant |
| Q10 | 2 | Standard treatments not found |

With:
- 1 = bad
- 5 = excellent

**Definitions:**
- **Retrieval:** Have the correct documents been found?

# Answer Evaluation

In [16]:
results_answer = []

for item in questions:
    
    question = item["question"]
    print(f'{question:-^80}')

    output = ask_medical_assistant(
        question,
        retriever
    )

    results_answer.append(output)
    print("\n\n")

-----------------------------What is glioblastoma?------------------------------
Original query: What is glioblastoma?
Processed query: glioblastoma


KeyboardInterrupt: 

### V1

| Question  | Relevance | Faithfulness | Clarity |
|:-----|:-----:|:-----:|:-----:|
| Q1 | 5 | 5 | 5 |
| Q2 | 4 | 4 | 3 |
| Q3 | 2 | 4 | 4 |
| Q4 | 5 | 5 | 4 |
| Q5 |  4 | 4 | 4 |
| Q6 |  3 | 3 | 4 |
| Q7 |  2 | 4 | 4 |
| Q8 |  5 | 5 | 4 |
| Q9 |  1 | 4 | 3 |
| Q10 |  1 | 5 | 2 |

With:
- 1 = bad
- 5 = excellent



**Definitions:**
- **Relevance:** Does it answer the question?
- **Accuracy:** Is it accurate in the context?
- **Clarity:** Is it understandable?


Identified limitations:
- Duplicate documents
- Overly simplistic retrieval
- Insufficient model

### V2

| Question  | Relevance | Faithfulness | Clarity |
|:-----|:-----:|:-----:|:-----:|
| Q1 | 5 | 5 | 5 |
| Q2 | 4 | 4 | 3 |
| Q3 | 2 | 4 | 4 |
| Q4 | 5 | 5 | 4 |
| Q5 |  4 | 4 | 4 |
| Q6 |  3 | 3 | 4 |
| Q7 |  2 | 4 | 4 |
| Q8 |  5 | 5 | 4 |
| Q9 |  1 | 4 | 3 |
| Q10 |  1 | 5 | 2 |

With:
- 1 = bad
- 5 = excellent



**Definitions:**
- **Relevance:** Does it answer the question?
- **Accuracy:** Is it accurate in the context?
- **Clarity:** Is it understandable?

In [ ]:
qdeqe

# Dowload Results

In [ ]:
rows = []

for question, retrieval, answer in zip(
    questions,
    results_retriever,
    results_answer
):

    docs = retrieval["documents"][0]
    metas = retrieval["metadatas"][0]
    distances = retrieval["distances"][0]

    rows.append({
        "question": question["question"],

        "answer": answer,

        "doc_1": docs[0],
        "doc_2": docs[1],
        "doc_3": docs[2],

        "pmid_1": metas[0]["pmid"],
        "pmid_2": metas[1]["pmid"],
        "pmid_3": metas[2]["pmid"],

        "distance_1": distances[0],
        "distance_2": distances[1],
        "distance_3": distances[2],
    })

df_eval = pd.DataFrame(rows)
df_eval.head()

In [ ]:
df_eval["retrieval"] = None
df_eval["retrieval_comments"] = ""

df_eval["relevance"] = None
df_eval["faithfulness"] = None
df_eval["clarity"] = None

In [ ]:
df_eval["retrieval"] = [4,3,2,3,1,2,3,2,2,1]
df_eval["retrieval_comments"] = [
    "",
    "",
    "",
    "Confusion btw glioma & glioblastoma",
    "Not Relevant",
    "Relevance ?",
    "",
    "Only doc 1 is relevant",
    "Doc 1 : not relevant",
    "Confusion btw glioma & glioblastoma"
]

In [ ]:
df_eval["relevance"] = [5,4,2,5,4,3,2,5,1,1]
df_eval["faithfulness"] = [5,4,4,5,4,3,4,5,4,5]
df_eval["clarity"] = [5,3,4,4,4,4,4,4,3,2]

In [ ]:
df_eval.head()

In [ ]:
version = 1
model = "qwen2.5:1.5b"
step = "baseline"

df_eval.to_csv(
    f"../data/evaluations/V{version}_{step}_evalN7_retriev_&_answ_model-{model}.csv",
    index=False
)

___

In [ ]:
summary = {
    "retrieval_mean": df_eval["retrieval"].mean(),
    "relevance_mean": df_eval["relevance"].mean(),
    "faithfulness_mean": df_eval["faithfulness"].mean(),
    "clarity_mean": df_eval["clarity"].mean(),
}

In [ ]:
print(summary)

Means:

| Retrieval  | Relevance | Faithfulness | Clarity |
|:-----|:-----:|:-----:|:-----:|
| 2.3 | 3.2 | 4.3 | 3.7 |

In [ ]:
df_eval.to_csv(
    f"../data/evaluations/V{version}_{step}_evalN7_summary_model-{model}.csv",
    index=False
)